# Session 15: Practice Quiz

**Notebook 2 of 2: Self-check** · about 30 minutes

For each question:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded. It runs offline, with no model and no network.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

print("ready")

## Q1 · Predict: four questions, four outcomes

The assistant retrieves with `top_k=4` and `min_score=0.6`. For each case, predict: is the answer `found`? Which sources are returned? Was the model called? What HTTP status does `POST /ask` return?

1. The best chunk scores 0.52.
2. Four chunks score above 0.6; the model returns `answerable: false`.
3. Four chunks score above 0.6; the model answers *"Promote db-03 [1]. Then rebuild it [5]."* with `citations: [1, 5]`.
4. Four chunks score above 0.6; the model answers *"Stop writes [2], then promote db-03 [1]."* with `citations: [1]`.

<details>
<summary>Show answer</summary>

| Case | found | Sources returned | Model called? | Status |
|---|---|---|---|---|
| 1 | false | none | **no**: nothing reached the threshold, so no tokens are spent | 200 |
| 2 | false | none | yes, and its tokens are reported | 200 |
| 3 | n/a | n/a | yes | **502**: `[5]` doesn't exist (only four sources were given), so the answer is refused with `InvalidAnswerError` |
| 4 | true | **1 and 2**, numbered 1 and 2 | yes | 200 |

Case 4 is the subtle one: the `citations` list forgot `[2]`, but the text cites it, and the text is what people read. Counting both makes the returned sources match what the answer says.

Case 3 is the reason for checking at all: "Then rebuild it" has no source the user could check. Showing it with a citation would make an unsupported claim look verified.

</details>

## Q2 · Debugging: the slow, forgetful UI

A first version of the Streamlit page:

```python
import streamlit as st
from knowledge_assistant.assistant import build_assistant

assistant = build_assistant()
question = st.text_input("Question")
if st.button("Ask"):
    answer = assistant.ask(question)
    st.markdown(answer.answer)
show_scores = st.checkbox("Show scores")
```

Users report two problems: every click takes about three seconds before anything happens, and when they tick "Show scores" after getting an answer, the answer disappears. Explain both, and fix them.

<details>
<summary>Show answer</summary>

**Both come from Streamlit's execution model: every interaction reruns the whole script.**

1. **Slow:** `build_assistant()` runs on every rerun: it loads the embedding model and opens the index each time. Cache it:

```python
@st.cache_resource
def get_assistant():
    return build_assistant()
```

`cache_resource` keeps one object per server process, shared by all sessions. That's right for a model and a database client, which are expensive and safe to share. (`st.cache_data` is for data you'd copy, such as a DataFrame.)

2. **Forgetful:** `st.button` is `True` only in the rerun its click caused. Ticking the checkbox causes another rerun, where the button is `False`, so the `if` block, and the answer, are skipped. Keep the answer in `st.session_state`, which survives reruns for that user:

```python
if st.button("Ask"):
    st.session_state["answer"] = get_assistant().ask(question)
if "answer" in st.session_state:
    st.markdown(st.session_state["answer"].answer)
```

The workbook's UI doesn't need the second fix (it has no other widgets), but the first one is a requirement, and a test checks it.

</details>

## Q3 · Debugging: the outage that improved the score

During a nightly evaluation, the LLM provider was down for the whole run. The report said:

```text
answer rate         0%
rejection rate    100%   unanswerable questions declined
```

The teammate's code:

```python
@property
def correctly_rejected(self) -> bool:
    return not self.item.answerable and not self.found
```

What went wrong, and what should the report have said?

<details>
<summary>Show answer</summary>

Every question failed with `GeneratorUnavailable`, so `found` stayed `False` for all of them, including the unanswerable ones. The code counts "not found" as a correct rejection without asking **why** nothing was found. An outage looks like perfect judgement.

Require that there was no error:

```python
return not self.item.answerable and not self.found and self.error is None
```

The report should have said: **rejection rate 0%, errors 20**. Errors aren't an outcome of the assistant's judgement: report them separately, and treat a run with errors as incomplete rather than comparing its rates with other runs. The provided CLI exits with code 1 when any question errored, for exactly this reason.

The general lesson for metrics: a metric that improves when the system breaks is measuring the wrong thing. Ask "what would this number be if everything failed?" for each one.

</details>

## Q4 · Engineering judgment: shipping a threshold

The notebook's measurements showed that `min_score=0.65` rejected all four unanswerable questions without losing an answerable one, while 0.60 let *"How many vacation days do new employees get?"* through. A teammate opens a pull request changing the default to 0.65, titled *"Fix: 100% rejection rate"*. How do you review it?

<details>
<summary>Show answer</summary>

Points a good review makes:

- **The evidence is circular.** 0.65 was chosen by looking at these four questions, so 100% on them proves little. Ask for questions that weren't used to choose it: write ten more unanswerable ones (realistic ones, close to the documentation's topics: on-call pay, office opening hours, the Wi-Fi password) and ten more answerable ones, and measure again.
- **The margin is thin.** The lowest answerable score was 0.684 and the highest unanswerable one 0.627. A threshold of 0.65 sits about 0.03 from each. A new document, a different chunk size or a rephrased question moves scores by more than that.
- **It's not the only defence.** With a real model, the vacation question reaches the LLM, whose `answerable` flag should decline it, because the rota policy talks about public holidays, not vacation days. Measure that before raising the threshold, since raising it costs answerable questions (at 0.70 one is already lost).
- **The title is wrong.** It isn't a fix: it's a trade-off between answering more and refusing more. The description should show the before-and-after table for all four rates.

A reasonable outcome: keep 0.6 as the default for now, add the new questions to the evaluation set, and revisit with a model-backed run.

</details>

## Q5 · Implementation: test the error mapping

Here's a small API that maps errors to status codes. Write pytest tests proving that a carrier outage gives 503 with a short message and no traceback, and that an unknown parcel gives 404. Use `TestClient`, and make the outage happen without a real carrier.

In [ ]:
class CarrierUnavailable(Exception):
    pass


class Carrier:
    def status(self, parcel_id: str) -> str:
        return "out for delivery"


def create_app(carrier: Carrier) -> FastAPI:
    app = FastAPI()

    @app.get("/parcels/{parcel_id}")
    def parcel(parcel_id: str) -> dict:
        if not parcel_id.startswith("P-"):
            raise HTTPException(404, "no such parcel")
        try:
            return {"parcel": parcel_id, "status": carrier.status(parcel_id)}
        except CarrierUnavailable as error:
            raise HTTPException(503, "the carrier is unavailable; try again") from error

    return app


print(TestClient(create_app(Carrier())).get("/parcels/P-1").json())

<details>
<summary>Show answer</summary>

```python
class DownCarrier(Carrier):
    def status(self, parcel_id: str) -> str:
        raise CarrierUnavailable("connect timeout to carrier-api.internal:8443")


def test_a_carrier_outage_is_503_without_details():
    client = TestClient(create_app(DownCarrier()))

    response = client.get("/parcels/P-1")

    assert response.status_code == 503
    assert response.json() == {"detail": "the carrier is unavailable; try again"}
    assert "carrier-api.internal" not in response.text  # the internal error stays internal
    assert "Traceback" not in response.text


def test_an_unknown_parcel_is_404():
    response = TestClient(create_app(Carrier())).get("/parcels/X-9")
    assert response.status_code == 404
```

What makes these good tests:

- **The fake is injected,** exactly like the workbook's scripted generator: `create_app` receives its dependency, so a test can pass one that fails on purpose. No patching, no network.
- **The fake's message contains something secret-looking** (an internal hostname), and the test proves it doesn't reach the client.
- **It checks the exact body,** not just the status, because clients and people read the message.

The workbook's API tests do the same with `ScriptedGenerator.error`.

</details>

## Q6 · Comparison: Streamlit versus the UI frameworks you know

You've built UIs with React, Angular, Blazor or server-side MVC (Spring MVC, ASP.NET MVC). Compare Streamlit with one of them on: where state lives, what runs when a user clicks, and when you would **not** choose Streamlit.

<details>
<summary>Show answer</summary>

| | React (or Angular, Blazor) | Server-side MVC | Streamlit |
|---|---|---|---|
| What runs on a click | An event handler, in the browser | A controller action for that request | **Your whole Python script, again, on the server** |
| Where state lives | Component state, in the browser | The database, the session, hidden fields | `st.session_state` per user; `st.cache_*` across users |
| What you write | Components, events, an API to call | Routes, controllers, templates | A top-to-bottom script; widgets return values |
| Who it suits | Product UIs | Websites and forms | Internal tools, demos, data apps, by people who write Python |

**Choose Streamlit** for an internal tool like this one: a few widgets, one page, built in an hour by the engineers who own the logic.

**Don't choose it** for a public product UI that needs fine-grained control over layout, routing, accessibility and performance; for pages with many users doing heavy work on each rerun (every interaction re-executes the script on the server); or when frontend developers will maintain it. There, the FastAPI service you built is the right foundation, with a proper frontend calling `POST /ask`. That's one reason the capstone keeps all logic out of `ui.py`: replacing the UI later costs nothing.

</details>